In [ ]:
%pip install python-dotenv langchain langchain-classic langchain_core langchain-tavily langchain-community langchain-openai openai langchain-google-genai langchain-anthropic

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

anthropic_key = os.getenv("ANTHROPIC_API_KEY")
print(anthropic_key[:5])

In [ ]:
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_anthropic import ChatAnthropic
from langchain.tools import tool


class contactInfo(BaseModel):
    name: str
    phone: str
    email: str

anthropic_model = ChatAnthropic(
    model="claude-sonnet-4-5",
    temperature=0
).with_structured_output(contactInfo)
 

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract contact information from the following text."),
    ("user", "{input}")
])  

chain =  prompt | anthropic_model

result = chain.invoke({"input": "What is phone 123-456-7890 of the person named John Doe and email john@example.com?"})

print(result)



In [ ]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain.tools import tool


class contactInfo(BaseModel):
    name: str
    phone: str
    email: str


anthropic_model = ChatAnthropic(model="claude-sonnet-4-5",
    max_retries=2, # Will wait and try again automatically
    temperature=0) 

# gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",
#     max_retries=6, # Will wait and try again automatically
#     temperature=0) 

@tool 
def search(tools: str) -> str:
    """Search for information."""
    return f"Results for: {tools}"

agent = create_agent(model=anthropic_model, 
        tools=[search],
        response_format=ToolStrategy(contactInfo))  
          

result = agent.invoke({"messages": [{"role": "user", 
    "content":"What is phone 123-456-7890 of the person named John Doe and email john@example.com?"}]})


print(result)
print("======")
print(result["structured_response"])

